In [1]:
import hydra
from omegaconf import DictConfig
from hydra import initialize, compose

import pandas as pd
import os

In [2]:
# Initialize the Hydra config within Jupyter
initialize(config_path="../configs")  # Point to your config directory

# Compose the configuration
cfg = compose(config_name="config")   # Load your main config.yaml

cfg

/tmp/ipykernel_1301149/560350757.py:2: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  initialize(config_path="../configs")  # Point to your config directory


{'validation': {'if_validate': False, 'batch_size': 8, 'save_model_every_n_epochs': 250, 'check_val_every_n_epochs': 100, 'check_val_monitor': 'val_loss/absolute_rssd', 'save_top_k_models': 40, 'early_stopping': True, 'early_stopping_patience': 100}, 'test': {'save_dir': './', 'checkpoints_parent_dir': None, 'checkpoints_name_list': 'all', 'batch_size': 1, 'epoch_index': None, 'checkpoint_path': None, 'test_save_parent_path': None, 'checkpoint_name': 'all'}, 'general': {'name': 'MERFISH_mouse_cortex', 'wandb': 'online', 'mode': 'train_and_test', 'seed': 0, 'enable_progress_bar': True, 'local_saved_path': None}, 'train': {'n_epochs': 1000, 'batch_size': 6, 'lr': 0.0005, 'fast_dev_run': False, 'weight_decay': 1e-12}, 'model': {'n_layers': 8, 'diffusion_noise_schedule': 'cosine', 'diffusion_steps': 1000, 'nu': {'p': 2}, 'hidden_mlp_dims': {'X': 256, 'y': 256, 'pos': 64}, 'hidden_dims': {'dx': 256, 'dy': 1, 'num_heads': 16, 'dim_ffX': 256, 'dim_ffy': 256, 'dd': 64, 'output_features_to_pos_

In [ ]:
## The following is the configuration file that you need to use for the experiment. Here is only for information. The command to run the experiment is in the next cell.

from datetime import datetime

# Get current date and time
now = datetime.now()

# Format it as a string
timestamp_str = now.strftime("%Y-%m-%d %H:%M:%S")

cfg.general.name = '10xgenomics_alzheimers' + '_' + timestamp_str
train_test_data_folder = 'train_test_split_1'

cfg.dataset.gene_columns_start = 13
cfg.dataset.gene_columns_end = 360
cfg.distribute.gpus_per_node=[6] # Change this to the number of GPUs you want to use
# cfg.general.wandb='disabled'

root_directory = '/home/anagupta/luna/' + train_test_data_folder
cfg.dataset.train_data_path = root_directory + '/train_data.csv' # Change this to the path of the train csv file
cfg.dataset.test_data_path = root_directory + '/test_data.csv' # Change this to the path of the test csv file
cfg.dataset.slice_images_path = root_directory + '/slice_images' # Change this to the path of the slice images
cfg.dataset.train_cell_images_path = root_directory + '/train_cell_images' # Change this to the path of the train cell images
cfg.dataset.test_cell_images_path = root_directory + '/test_cell_images' # Change this to the path of the test cell images
cfg.test.save_dir = root_directory + '/10xgenomics_alzheimers_test_results' # Change this to the directory where you want to save the results
cfg.train.batch_size=6

cfg.dataset.maximum_graph_size.train=5000
cfg.dataset.maximum_graph_size.test=5000

from omegaconf import OmegaConf

# Save the cfg configuration file
OmegaConf.save(cfg, root_directory + '/config.yaml')

In [15]:
HYDRA_FULL_ERROR=1  
output_path =  root_directory + '/output.txt'
!python3 /home/anagupta/luna/LUNA/main.py --config-path=$root_directory --config-name=config.yaml > $output_path

Seed set to 0
/home/anagupta/luna/LUNA/datasets/data_module.py:106: FutureWarning: Index.is_numeric is deprecated. Use pandas.api.types.is_any_real_numeric_dtype instead
  if not self.input_data.index.is_numeric():
/home/anagupta/luna/.venv/lib/python3.10/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)
/home/anagupta/luna/LUNA/datasets/data_module.py:106: FutureWarning: Index.is_numeric is deprecated. Use pandas.api.types.is_any_real_numeric_dtype instead
  if not self.input_data.index.is_numeric():
/home/anagupta/luna/.venv/lib/python3.10/site-packages/torch_geometric/data/in_memory_dataset.py:300: Use

In [14]:
import tarfile
from PIL import Image
from io import BytesIO
import os
import numpy as np

extracted_folder = '/home/anagupta/luna/preprocessed_images/cell_imagess'

extracted_images = []

# loop through the extracted folder and get all the tar files
for root, dirs, files in os.walk(extracted_folder):
    for file in files:
        if file.endswith('.tar'):
            tar_file_path = os.path.join(root, file)
            # print(tar_file_path)
            # Open the tar file
            with tarfile.open(tar_file_path, 'r') as tar:
                # Loop through the files in the tar archive
                for tarinfo in tar:
                    # Check if the file inside the tar is an image
                    # print(tarinfo.name)
                    if tarinfo.name.endswith(('.npy')):
                        # Extract the file as a BytesIO object (in memory)
                        file_obj = tar.extractfile(tarinfo)
                        image_data = file_obj.read()
                        # Convert the image data to a PIL Image
                        image_array = np.load(BytesIO(image_data))
                        extracted_images.append(image_array)

print("Number of images extracted:", len(extracted_images))


Number of images extracted: 53912
